# expiMap on cancer data (Colab)

In [ ]:
# 1. connect Google Drive FIRST: all results and downloaded data are kept there
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/expimap_cancer'
import os; os.makedirs(DRIVE, exist_ok=True); print('results go to', DRIVE)

In [ ]:
# 2. check the GPU
!nvidia-smi | head -n 12
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# 3. get the code
REPO = 'https://github.com/Hastikabirsamadi/expiMap-cancer-pathways.git'
%cd /content
!git clone $REPO repo || git -C repo pull
%cd repo

In [ ]:
# 4. install packages and get the PBMC reference data and Reactome pathways
# if `import torch` breaks afterwards: Runtime > Disconnect and delete runtime, then rerun from the top
%pip install -q -r requirements-colab.txt
!python download_data.py

## CTCL (blood cancer)
Downloads the CTCL atlas blood file, keeps PBMC samples only (at most 1000 cells per donor), and saves the small subsample to Drive. After the first run, the download and subsampling are skipped.

In [ ]:
# 5. download the full file to Colab's local disk (3.4 GB, not to Drive), then subsample to Drive
import os
URL = 'https://datasets.cellxgene.cziscience.com/ca954737-f273-4144-b29d-1dccdbd22e4c.h5ad'
FULL = '/content/ctcl_blood.h5ad'
SUB = f'{DRIVE}/ctcl_pbmc_subsample.h5ad'
if not os.path.exists(SUB):
    !wget -c -q --show-progress -O $FULL $URL
    !python prepare_ctcl.py --h5ad $FULL --out $SUB --per_donor 1000
else:
    print('subsample already on Drive:', SUB)

In [ ]:
# 6. map CTCL onto the PBMC reference by surgery (about the time of one baseline run)
# REF_MODEL: the trained PBMC reference from the baseline project (unmodified expiMap, seed 0), read straight from Drive.
# If this folder had no model, a new one would be trained and saved there.
REF_MODEL = '/content/drive/MyDrive/expimap_results/runs/baseline/seed0/ref_model'
OUT = f'{DRIVE}/ctcl_map/seed0'
!python map_ctcl.py --ctcl $SUB --ref_model $REF_MODEL --out $OUT --seed 0

In [ ]:
# 7. look at the results
# the figures are redrawn from the saved results (no training), so this also updates an older run
OUT = f'{DRIVE}/ctcl_map/seed0'
!python plot_ctcl.py --out $OUT
import json, pandas as pd
from IPython.display import Image, display
print(json.dumps(json.load(open(f'{OUT}/metrics.json')), indent=1))
for f in ['fig1_cell_types', 'fig2_disease', 'fig3_author_labels', 'fig4_confidence', 'fig5_mixing']:
    display(Image(f'{OUT}/{f}.png'))
display(pd.read_csv(f'{OUT}/crosstab_normal.csv', index_col=0))

## Part A: which pathways change in the cancer cells
Uses the saved mapping (no training, no GPU needed). Compares malignant with non-malignant CD4+ T cells inside each patient, and MF with Sezary inside the studies that have both. Results go to `ctcl_map/seed0/partA`.

In [ ]:
# 8. Part A (needs cells 1, 3 and 4 after a disconnect)
OUT = f'{DRIVE}/ctcl_map/seed0'
!python pathways_ctcl.py --map $OUT
import pandas as pd
from IPython.display import Image, display
A = f'{OUT}/partA'
display(pd.read_csv(f'{A}/comp1_malignant_vs_cd4.csv').head(20))
for f in ['fig_A1_malignant_vs_cd4_heatmap', 'fig_A2_pathway_umaps', 'fig_A3_celltype_profiles', 'fig_A4_mf_vs_sezary']:
    try: display(Image(f'{A}/{f}.png'))
    except FileNotFoundError: print('not made:', f)